# AI風險管理

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明 AI 專案常見風險類型：資料品質、模型穩定性、組織人力、法規合規與侵權風險。
2. 使用簡單資料分析方法辨識資料遺漏、類別偏差與標註異常。
3. 以模型表現差異觀察資料漂移或模型漂移對 AI 系統的影響。
4. 建立一份簡化版 AI 風險登錄表，並依風險機率與衝擊程度排序。
5. 將法規合規需求轉換成可檢查的治理清單。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立後續範例會用到的隨機種子設定。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import entropy
from collections import Counter

np.random.seed(42)
print('環境設定完成')


## 核心概念說明

AI 風險管理不是模型上線後才補做的文件工作，而是從規劃、資料蒐集、模型訓練、測試、部署到維運都要持續進行的治理流程。

本章可將 AI 風險分為五大類：

| 風險類型 | 說明 | 常見例子 |
|---|---|---|
| 資料品質風險 | 資料缺失、錯誤、不具代表性或標註錯誤 | 缺失值、資料偏頗、不正確標註 |
| 模型穩定性風險 | 模型在環境變化後表現下降 | 模型漂移、資料漂移、概念漂移、過擬合 |
| 組織人力風險 | 導入 AI 時跨部門協作與技能不足 | 角色不清、技能短缺、變革抵制 |
| 法規合規風險 | 個資、隱私、產業規範未被遵守 | GDPR、台灣個資法、HIPAA、PCI-DSS |
| 侵權風險 | 未授權使用第三方資料、模型、程式碼或演算法 | 訓練資料侵權、開源授權違規 |

接下來的實作會用簡化資料模擬企業導入 AI 信用風險模型時，如何檢查資料偏誤、模型漂移與合規風險。


In [ ]:
# ── 示範：資料品質與偏誤檢查 ────────────────────────────
# 這段程式碼建立一份模擬信用評分資料，檢查缺失值、群體分布不均與核准率差異，對應資料品質風險與資料偏頗風險。

import numpy as np
import pandas as pd

np.random.seed(42)
n = 300
customer_group = np.random.choice(['A群體', 'B群體', 'C群體'], size=n, p=[0.65, 0.25, 0.10])
income = np.random.normal(60000, 15000, n).round(0)
debt_ratio = np.random.beta(2, 5, n).round(2)

# 模擬偏誤：C群體即使條件相近，也較容易被拒絕
approval_score = income / 100000 - debt_ratio
approval_score -= np.where(customer_group == 'C群體', 0.18, 0)
approved = (approval_score > 0.25).astype(int)

# 製造少量缺失值
income[np.random.choice(n, 12, replace=False)] = np.nan

df = pd.DataFrame({
    'customer_group': customer_group,
    'income': income,
    'debt_ratio': debt_ratio,
    'approved': approved
})

missing_report = df.isna().sum()
group_distribution = df['customer_group'].value_counts(normalize=True).round(3)
approval_by_group = df.groupby('customer_group')['approved'].mean().round(3)

print('缺失值檢查：')
print(missing_report)
print('\n群體分布比例：')
print(group_distribution)
print('\n各群體核准率：')
print(approval_by_group)

risk_flags = []
if missing_report['income'] > 0:
    risk_flags.append('資料遺漏風險：income 欄位存在缺失值')
if group_distribution.min() < 0.15:
    risk_flags.append('資料代表性風險：至少一個群體樣本比例低於 15%')
if approval_by_group.max() - approval_by_group.min() > 0.2:
    risk_flags.append('公平性風險：不同群體核准率差距超過 20 個百分點')

print('\n偵測到的風險：')
for flag in risk_flags:
    print('-', flag)


## 資料漂移與模型漂移

AI 系統上線後，外部環境可能改變，例如客戶行為、經濟條件、設備狀態或資料來源格式改變。這會造成輸入資料分布與訓練期間不同，稱為資料漂移。

若模型因資料或概念變化而預測準確度下降，則可能出現模型漂移。實務上可透過以下方式監控：

- 定期比較訓練資料與近期資料的分布差異。
- 追蹤模型準確率、召回率、錯誤率等指標。
- 設定警戒門檻，超過門檻時啟動重訓、人工審查或風險通報。

以下範例用簡化的信用核准模型示範：當新資料的收入分布與負債比例改變後，模型表現可能下降。


In [ ]:
# ── 示範：模型穩定性與資料漂移監控 ─────────────────────────
# 這段程式碼訓練一個簡單分類模型，接著用分布已改變的新資料測試，觀察模型準確率下降與資料漂移指標。

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.stats import entropy

np.random.seed(7)

# 訓練期間資料
n_train = 500
train_income = np.random.normal(65000, 12000, n_train)
train_debt = np.random.beta(2, 5, n_train)
train_y = ((train_income / 100000 - train_debt) > 0.25).astype(int)
train_X = pd.DataFrame({'income': train_income, 'debt_ratio': train_debt})

# 上線後資料：收入下降、負債比例提高，模擬環境改變
n_new = 300
new_income = np.random.normal(54000, 14000, n_new)
new_debt = np.random.beta(3, 4, n_new)
new_y = ((new_income / 100000 - new_debt) > 0.25).astype(int)
new_X = pd.DataFrame({'income': new_income, 'debt_ratio': new_debt})

model = LogisticRegression()
model.fit(train_X, train_y)

train_acc = accuracy_score(train_y, model.predict(train_X))
new_acc = accuracy_score(new_y, model.predict(new_X))

# 使用直方圖機率分布與 KL divergence 作為簡化漂移指標
bins = np.linspace(20000, 100000, 12)
train_hist, _ = np.histogram(train_income, bins=bins, density=True)
new_hist, _ = np.histogram(new_income, bins=bins, density=True)
train_hist = train_hist + 1e-9
new_hist = new_hist + 1e-9
kl_divergence = entropy(new_hist, train_hist)

print(f'訓練資料準確率：{train_acc:.3f}')
print(f'上線後新資料準確率：{new_acc:.3f}')
print(f'收入分布 KL divergence：{kl_divergence:.3f}')

if new_acc < train_acc - 0.08:
    print('警示：模型表現明顯下降，可能存在模型漂移。')
if kl_divergence > 0.15:
    print('警示：輸入資料分布明顯改變，可能存在資料漂移。')


## AI 風險登錄表

漂移只是眾多風險之一。實務上會用風險登錄表把資料品質、公平性、模型穩定性、法規遵循等風險一起列管，以「風險分數 = 發生機率 × 衝擊程度」排序，決定優先處理順序。


In [ ]:
# ── 實際應用：AI 風險登錄表排序 ─────────────────────────
# 這段程式碼建立簡化版 AI 風險登錄表，使用風險分數等於機率乘以衝擊程度，協助專案團隊排序優先處理項目。

import pandas as pd

risk_register = pd.DataFrame([
    {'risk': '資料缺失導致模型誤判', 'category': '資料品質', 'probability': 4, 'impact': 5, 'control': '建立缺失值檢查與資料補值流程'},
    {'risk': '特定群體核准率偏低', 'category': '偏誤公平性', 'probability': 3, 'impact': 5, 'control': '定期檢查群體表現差異並進行公平性審查'},
    {'risk': '模型上線後準確率下降', 'category': '模型穩定性', 'probability': 4, 'impact': 4, 'control': '設定模型監控儀表板與重訓門檻'},
    {'risk': '未取得個資使用同意', 'category': '法規合規', 'probability': 2, 'impact': 5, 'control': '導入同意管理與資料處理紀錄'},
    {'risk': '使用未授權第三方資料', 'category': '侵權風險', 'probability': 2, 'impact': 4, 'control': '建立資料來源與授權條款盤點'},
    {'risk': '業務與技術部門需求認知不同', 'category': '組織人力', 'probability': 4, 'impact': 3, 'control': '定義角色責任並建立跨部門例會'}
])

risk_register['risk_score'] = risk_register['probability'] * risk_register['impact']
risk_register['priority'] = pd.cut(
    risk_register['risk_score'],
    bins=[0, 8, 14, 25],
    labels=['低', '中', '高']
)

result = risk_register.sort_values('risk_score', ascending=False).reset_index(drop=True)
result
